In [ ]:
import os
import logging
import time

from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date

import pandas as pd
import requests



# ENDPOINTS RE/MAX


API_URL = (
    "https://api-ar.redremax.com/remaxweb-ar/api/listings/"
    "findAllWithEntrepreneurships"
)

DETAIL_URL_TEMPLATE = (
    "https://api-ar.redremax.com/remaxweb-ar/api/listings/"
    "findBySlug/{slug}"
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json",
    "Referer": "https://www.remax.com.ar/"
}


logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s"
)

logger = logging.getLogger(__name__)



# FUNCIONES AUXILIARES


def safe_get_value(item, key):
    """
    Extrae de forma segura el valor de claves que pueden
    ser diccionarios o None.
    """

    if not isinstance(item, dict):
        return None

    val = item.get(key)

    if isinstance(val, dict):
        return val.get("value") or val.get("name")

    return val



# LISTADO GENERAL


def get_page(page, page_size):
    """Consulta una página del listado general de propiedades."""

    params = {
        "page": page,
        "pageSize": page_size,
        "sort": "-createdAt",
        "in": "operationId:1",
        "locations": "in:CF@<b>Capital</b> <b>Federal</b>::::::",
        "landingPath": ""
    }

    try:
        response = requests.get(
            API_URL,
            params=params,
            headers=HEADERS,
            timeout=30
        )

        logger.info(
            f"Página {page + 1} | Status: {response.status_code}"
        )

        response.raise_for_status()

        return response.json()

    except requests.exceptions.RequestException as e:
        logger.error(
            f"Error en página {page + 1}: {e}"
        )

    except ValueError as e:
        logger.error(
            f"Error interpretando JSON en página {page + 1}: {e}"
        )

    return None



# DETALLE DE CADA PROPIEDAD


def get_listing_detail(
    slug,
    property_id=None,
    max_retries=2
):
    """
    Consulta el detalle individual utilizando el slug.

    Si la consulta falla, se realizan reintentos antes de
    devolver un diccionario vacío.
    """

    if not slug:
        logger.warning(
            f"Propiedad {property_id} sin slug — "
            "no se puede pedir el detalle"
        )
        return {}

    url = DETAIL_URL_TEMPLATE.format(slug=slug)

    for intento in range(1, max_retries + 1):

        try:
            res = requests.get(
                url,
                headers=HEADERS,
                timeout=10
            )

            if res.status_code == 200:

                data = res.json()

                if isinstance(data, dict):
                    return data.get("data") or data

                return {}

            if res.status_code == 429:

                logger.warning(
                    f"429 (rate limit) en detalle de {slug}, "
                    f"esperando antes de reintentar "
                    f"(intento {intento})"
                )

                time.sleep(2 * intento)
                continue

            logger.warning(
                f"Status {res.status_code} "
                f"en detalle de {slug} "
                f"(intento {intento})"
            )

        except requests.exceptions.RequestException as e:

            logger.warning(
                f"Error de conexión en detalle de {slug} "
                f"(intento {intento}): {e}"
            )

    logger.error(
        f"No se pudo obtener el detalle de {slug} "
        f"después de {max_retries} intentos"
    )

    return {}



# PARSEO DE CADA PROPIEDAD


def parse_listing(item, detail):

    if not isinstance(item, dict):
        return None

    if not isinstance(detail, dict):
        detail = {}

    try:

        property_id = item.get("id")


        # CAMPOS ENRIQUECIDOS


        descripcion = (
            detail.get("description")
            or item.get("description")
            or ""
        )

        fecha_pub = (
            item.get("createdAt")
            or item.get("publicationDate")
            or detail.get("createdAt")
            or detail.get("publicationDate")
        )

        year_built = (
            detail.get("yearBuilt")
            or item.get("yearBuilt")
        )

        antiguedad = (
            date.today().year - year_built
            if year_built
            else None
        )

        apto_credito = detail.get("aptCredit")


        # AMENITIES


        features_raw = (
            detail.get("features")
            or item.get("features")
        )

        amenities = None

        if features_raw:

            valores = []

            for elemento in features_raw:

                if isinstance(elemento, dict):

                    valor = elemento.get(
                        "value",
                        elemento.get("name")
                    )

                    if valor:
                        valores.append(str(valor))

                else:
                    valores.append(str(elemento))

            if valores:
                amenities = " | ".join(valores)


        # PRECIO


        precio = item.get("price")
        moneda = safe_get_value(item, "currency")

        expensas = item.get("expensesPrice")

        moneda_expensas = safe_get_value(
            item,
            "expensesCurrency"
        )

        operacion = safe_get_value(
            item,
            "operation"
        )

        tipo_propiedad = safe_get_value(
            item,
            "type"
        )

        estado = safe_get_value(
            item,
            "listingStatus"
        )


        # UBICACIÓN


        geo_label = item.get("geoLabel")

        barrio = (
            geo_label.split(",")[0].strip()
            if geo_label
            else None
        )

        latitud = None
        longitud = None

        location = item.get("location")

        if isinstance(location, dict):

            coordinates = location.get(
                "coordinates",
                []
            )

            if (
                isinstance(coordinates, list)
                and len(coordinates) >= 2
            ):
                longitud = coordinates[0]
                latitud = coordinates[1]


        # SUPERFICIES


        superficie_total = (
            item.get("dimensionTotalBuilt")
            or item.get("dimensionLand")
        )

        superficie_cubierta = (
            item.get("dimensionCovered")
        )


        # PRECIO POR M2


        precio_m2_total = None
        precio_m2_cubierto = None

        if precio is not None:

            if (
                superficie_total
                and superficie_total > 0
            ):
                precio_m2_total = round(
                    precio / superficie_total,
                    2
                )

            if (
                superficie_cubierta
                and superficie_cubierta > 0
            ):
                precio_m2_cubierto = round(
                    precio / superficie_cubierta,
                    2
                )


        # LINK


        slug = item.get("slug")

        if slug:
            link = (
                "https://www.remax.com.ar/"
                f"listings/{slug}"
            )

        else:
            link = (
                "https://www.remax.com.ar/"
                f"listings/{property_id}"
            )


        # RESULTADO


        return {
            "ID": property_id,
            "ID_Entidad": item.get("entityId"),
            "Operacion": operacion,
            "Tipo_Propiedad": tipo_propiedad,
            "Titulo": item.get("title"),
            "Descripcion": descripcion,
            "Fecha_Publicacion": fecha_pub,
            "Antiguedad": antiguedad,
            "Apto_Credito": apto_credito,
            "Amenities": amenities,
            "Moneda": moneda,
            "Precio_Valor": precio,
            "Moneda_Expensas": moneda_expensas,
            "Expensas": expensas,
            "Precio_m2_Total": precio_m2_total,
            "Precio_m2_Cubierto": precio_m2_cubierto,
            "Ambientes": item.get("totalRooms"),
            "Dormitorios": item.get("bedrooms"),
            "Baños": item.get("bathrooms"),
            "Superficie_Total_m2": superficie_total,
            "Superficie_Cubierta_m2": superficie_cubierta,
            "Direccion": item.get("displayAddress"),
            "Barrio": barrio,
            "Ubicacion": geo_label,
            "Latitud": latitud,
            "Longitud": longitud,
            "Estado": estado,
            "ID_Interno": item.get("internalId"),
            "Link": link
        }

    except Exception as e:

        logger.error(
            "Error procesando propiedad "
            f"{item.get('id')}: {e}"
        )

        return None



# DETALLE EN PARALELO


def obtener_detalles_en_paralelo(
    listings,
    detail_workers=8,
    max_retries=2
):

    detalles = {}

    with ThreadPoolExecutor(
        max_workers=detail_workers
    ) as executor:

        futures = {
            executor.submit(
                get_listing_detail,
                item.get("slug"),
                item.get("id"),
                max_retries
            ): item.get("id")

            for item in listings

            if (
                isinstance(item, dict)
                and item.get("id")
            )
        }

        completados = 0

        for future in as_completed(futures):

            property_id = futures[future]

            try:
                detalles[property_id] = (
                    future.result()
                )

            except Exception as e:

                logger.error(
                    "Excepción obteniendo detalle "
                    f"de {property_id}: {e}"
                )

                detalles[property_id] = {}

            completados += 1

            if completados % 20 == 0:

                print(
                    f"      ...detalle "
                    f"{completados}/{len(futures)} "
                    "propiedades"
                )

    return detalles



# SCRAPER PRINCIPAL


def run_scraper(
    page_size=100,
    max_pages=200,
    detail_workers=8,
    detail_max_retries=2,
    sleep_time=1,
    output_dir="data/raw"
):

    os.makedirs(
        output_dir,
        exist_ok=True
    )

    all_data = []

    errores_propiedades = 0
    errores_paginas = 0
    paginas_exitosas = 0

    ids_obtenidos = set()

    print("\n" + "-" * 60)

    print(
        "INICIANDO SCRAPER RE/MAX — "
        "CABA COMPLETO (paralelo)"
    )

    print("-" * 60)

    print(
        f"Propiedades por página: "
        f"{page_size}"
    )

    print(
        f"Máximo de páginas: "
        f"{max_pages}"
    )

    print(
        f"Workers para detalle: "
        f"{detail_workers}"
    )

    print("-" * 60)


    # PAGINACIÓN


    for page in range(max_pages):

        print(
            f"\nProcesando página "
            f"{page + 1}..."
        )

        data = get_page(
            page,
            page_size
        )

        if data is None:

            errores_paginas += 1

            print(
                "No se pudo obtener página "
                f"{page + 1}"
            )

            time.sleep(3)

            continue

        listings = []

        if isinstance(data, dict):

            raw_data = data.get(
                "data",
                {}
            )

            if isinstance(raw_data, dict):

                listings = (
                    raw_data.get("listings")
                    or raw_data.get("data")
                    or []
                )

            elif isinstance(
                raw_data,
                list
            ):
                listings = raw_data

        if not listings:

            print(
                "\nNo hay más propiedades."
            )

            break

        paginas_exitosas += 1

        print(
            "   Propiedades recibidas: "
            f"{len(listings)}"
        )


        # DETALLE


        print(
            "   Pidiendo detalle de "
            f"{len(listings)} propiedades..."
        )

        t0 = time.time()

        detalles_por_id = (
            obtener_detalles_en_paralelo(
                listings,
                detail_workers,
                detail_max_retries
            )
        )

        print(
            "   Detalle completo en "
            f"{time.time() - t0:.1f}s"
        )

        vacios = sum(
            1
            for d in detalles_por_id.values()
            if not d
        )

        if vacios:

            print(
                f"   {vacios}/{len(listings)} "
                "propiedades quedaron sin detalle"
            )


        # PARSEO


        nuevas = 0

        for item in listings:

            property_id = (
                item.get("id")
                if isinstance(item, dict)
                else None
            )

            detail = detalles_por_id.get(
                property_id,
                {}
            )

            parsed = parse_listing(
                item,
                detail
            )

            if parsed:

                pid = parsed["ID"]

                if pid not in ids_obtenidos:

                    all_data.append(
                        parsed
                    )

                    ids_obtenidos.add(
                        pid
                    )

                    nuevas += 1

            else:
                errores_propiedades += 1

        print(
            f"   Propiedades nuevas: "
            f"{nuevas}"
        )

        print(
            f"   Total acumulado: "
            f"{len(all_data)}"
        )


        # ÚLTIMA PÁGINA


        if len(listings) < page_size:

            print(
                "\nÚltima página detectada."
            )

            break


        # BACKUP


        if (page + 1) % 10 == 0:

            backup_df = pd.DataFrame(
                all_data
            )

            backup_path = os.path.join(
                output_dir,
                f"backup_pagina_{page + 1}.csv"
            )

            backup_df.to_csv(
                backup_path,
                index=False,
                encoding="utf-8-sig"
            )

            print(
                f"Backup guardado: "
                f"{backup_path}"
            )

        time.sleep(
            sleep_time
        )


    # DATAFRAME FINAL


    if not all_data:

        print(
            "\nNO SE OBTUVIERON DATOS."
        )

        return None

    df = pd.DataFrame(
        all_data
    )


    # DUPLICADOS


    cantidad_antes = len(df)

    df = df.drop_duplicates(
        subset=["ID"]
    )

    duplicados = (
        cantidad_antes - len(df)
    )


    # VARIABLES NUMÉRICAS


    columnas_numericas = [
        "Precio_Valor",
        "Expensas",
        "Precio_m2_Total",
        "Precio_m2_Cubierto",
        "Ambientes",
        "Dormitorios",
        "Baños",
        "Superficie_Total_m2",
        "Superficie_Cubierta_m2",
        "Antiguedad",
        "Latitud",
        "Longitud"
    ]

    for columna in columnas_numericas:

        if columna in df.columns:

            df[columna] = pd.to_numeric(
                df[columna],
                errors="coerce"
            )


    # ORDEN DE COLUMNAS


    columnas = [
        "ID",
        "ID_Entidad",
        "Operacion",
        "Tipo_Propiedad",
        "Titulo",
        "Descripcion",
        "Fecha_Publicacion",
        "Antiguedad",
        "Apto_Credito",
        "Amenities",
        "Moneda",
        "Precio_Valor",
        "Moneda_Expensas",
        "Expensas",
        "Precio_m2_Total",
        "Precio_m2_Cubierto",
        "Ambientes",
        "Dormitorios",
        "Baños",
        "Superficie_Total_m2",
        "Superficie_Cubierta_m2",
        "Direccion",
        "Barrio",
        "Ubicacion",
        "Latitud",
        "Longitud",
        "Estado",
        "ID_Interno",
        "Link"
    ]

    df = df[
        [
            c
            for c in columnas
            if c in df.columns
        ]
    ]


    # EXPORTACIÓN


    csv_path = os.path.join(
        output_dir,
        "remax_caba_propiedades.csv"
    )

    xlsx_path = os.path.join(
        output_dir,
        "remax_caba_propiedades.xlsx"
    )

    df.to_csv(
        csv_path,
        index=False,
        encoding="utf-8-sig"
    )

    df.to_excel(
        xlsx_path,
        index=False
    )


    # RESUMEN


    print("\n" + "-" * 60)

    print(
        "SCRAPING FINALIZADO"
    )

    print("-" * 60)

    print(
        f"Páginas procesadas: "
        f"{paginas_exitosas}"
    )

    print(
        f"Registros obtenidos: "
        f"{len(df)}"
    )

    print(
        f"Duplicados eliminados: "
        f"{duplicados}"
    )

    print(
        f"Errores de propiedades: "
        f"{errores_propiedades}"
    )

    print(
        f"Errores de páginas: "
        f"{errores_paginas}"
    )

    print(
        f"\nCSV: {csv_path}"
    )

    print(
        f"Excel: {xlsx_path}"
    )

    print("-" * 60)

    return df